# Core types and their behaviour

Intermediate work with Python’s core types is less about memorising method names and more about understanding behaviour. Numbers, text, bytes, containers, and mappings all have trade-offs. Some values compare by contents, some by identity; some are immutable, some are not; some preserve order, some are specialised for lookup speed.

The goal of this module is to make those behaviours feel predictable. You should be able to reason about floating-point surprises, text encodings, and why one container fits a task better than another.

A good habit here is to ask not only “can Python do this?” but also “what type am I really working with, and what guarantees does it give me?”

## Visual model

```text
text input -> parse/convert -> correct type -> safe operation
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. Numbers

### `int` is arbitrary precision

```text
>>> 2 ** 1000
10715086071862673209484250490600018105614048117055336074437503883703510511249361...
>>> (2**64).bit_length()
65
```


No overflow, ever. Python promotes silently and without limit. This is a
genuine relief coming from C or Java — no `long long`, no wraparound bugs.

The cost is that ints are objects, not machine words:

```text
>>> import sys
>>> sys.getsizeof(0), sys.getsizeof(1), sys.getsizeof(2**100)
(28, 28, 44)
```


28 bytes for the number 1, against 8 for a C `int64`. That is the memory story
behind "use NumPy for numeric arrays" (Module 29): a list of a million Python
ints is roughly 40 MB; a NumPy `int64` array of the same is 8 MB, contiguous.

Since 3.11 there is a safety limit on `int`↔`str` conversion (default 4300
digits) to prevent quadratic-time denial of service. `sys.set_int_max_str_digits`
adjusts it if you genuinely need giant decimal output.

### `float` is IEEE 754, and it will lie to you

```text
>>> 0.1 + 0.2
0.30000000000000004
>>> 0.1 + 0.2 == 0.3
False
>>> from decimal import Decimal
>>> Decimal(0.1)
Decimal('0.1000000000000000055511151231257827021181583404541015625')
```


Binary floating point cannot represent 0.1 exactly, the same way decimal cannot
represent 1/3. This is not a Python flaw; it is IEEE 754 and it is true in
JavaScript, C, Java, and your calculator. Python is merely honest about it in
the REPL.

**Never compare floats with `==`.**

In [ ]:
import math
math.isclose(0.1 + 0.2, 0.3)                          # True
math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12)       # tune for your domain

Note the two tolerances. Relative tolerance is right for large numbers, absolute
for values near zero (where relative tolerance degenerates). If either matters,
set both.

Other float facts that bite:

```text
>>> round(2.5), round(3.5), round(0.5)
(2, 4, 0)                       # banker's rounding: ties go to even
>>> float('inf') > 10**1000
True
>>> float('nan') == float('nan')
False                           # NaN is not equal to itself. By design.
>>> math.isnan(x)               # the only correct NaN test
>>> 1e16 + 1 == 1e16
True                            # beyond 2**53, integers are not exact in float
```


Banker's rounding is deliberate: always rounding halves up biases sums upward.
It matches IEEE 754's default mode. If you need "round half up", use `Decimal`
with an explicit rounding mode.

### `Decimal` for money, `Fraction` for exactness

In [ ]:
from decimal import Decimal, ROUND_HALF_UP, getcontext

Decimal("0.1") + Decimal("0.2") == Decimal("0.3")     # True
Decimal("19.99") * 3                                   # Decimal('59.97') exactly

getcontext().prec = 28
Decimal("1.005").quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)  # 1.01

**Construct `Decimal` from strings, never from floats.** `Decimal(0.1)` faithfully
copies the float's error into the Decimal; `Decimal("0.1")` is exact.

**Use `Decimal` for money.** Or, better still, store money as an integer number
of minor units (cents) and format on display — that is what most payment systems
do, and it makes the arithmetic trivially exact.

In [ ]:
from fractions import Fraction
Fraction(1, 3) + Fraction(1, 6) == Fraction(1, 2)      # True, exactly

### `bool` is an `int`

```text
>>> True + True
2
>>> isinstance(True, int)
True
>>> sum([True, False, True])       # a legitimate idiom for counting matches
2
>>> sum(1 for x in data if x > 5)  # clearer, and the one to prefer
```


### Division and the operators worth knowing

In [ ]:
7 / 2       # 3.5    true division, ALWAYS float (even 4/2 -> 2.0)
7 // 2      # 3      floor division
-7 // 2     # -4     floors toward negative infinity, not toward zero
7 % 3       # 1
-7 % 3      # 2      the result carries the sign of the DIVISOR
divmod(7, 3)        # (2, 1)
7 ** 2      # 49

`-7 // 2 == -4` surprises people from C, where it truncates to `-3`. Python's
choice keeps the invariant `a == (a // b) * b + (a % b)` true for negatives,
which is what makes `%` useful for cyclic indexing: `-1 % 7 == 6`.

---

## Concept 2. Strings and bytes: the boundary that matters

This is the most important section in the module.

```text
str                                   bytes
"text, a sequence of characters"      b"data, a sequence of 0-255 integers"
        |                                      ^
        |  .encode('utf-8')                    |
        +--------------------------------------+
        ^                                      |
        |  .decode('utf-8')                    |
        +--------------------------------------+
```


- **`str` is text.** A sequence of Unicode code points. It has no encoding. It
  is what you compute with.
- **`bytes` is data.** A sequence of integers 0-255. It is what files, sockets,
  and disks actually hold.

Everything entering your program from outside is bytes. Everything leaving is
bytes. The rule, sometimes called the Unicode sandwich:

> **Decode at the input boundary. Work in `str`. Encode at the output boundary.**

In [ ]:
raw = b'\xc3\xa9clair'
text = raw.decode('utf-8')       # 'éclair'   <- decode ON THE WAY IN
print(len(raw), len(text))       # 8 6        <- bytes != characters
back = text.encode('utf-8')      # b'\xc3\xa9clair'  <- encode ON THE WAY OUT

Note `len(raw) != len(text)`. In UTF-8, one character can be 1 to 4 bytes. This
is why slicing bytes is dangerous and slicing str is safe.

### Always specify the encoding

In [ ]:
open("f.txt")                                 # uses the LOCALE default. A bug.
open("f.txt", encoding="utf-8")               # correct, always
Path("f.txt").read_text(encoding="utf-8")     # correct
open("f.bin", "rb")                           # binary: no encoding involved

The default encoding differs between Linux (usually UTF-8) and Windows (often
cp1252), which is the classic "works on my machine" file bug. Python 3.15 will
make UTF-8 the default; until then, be explicit. You can opt in early with
`PYTHONUTF8=1` or `python -X utf8`, and you can catch every unspecified-encoding
call with `python -X warn_default_encoding`.

### Handling bad bytes

In [ ]:
raw.decode('utf-8')                       # UnicodeDecodeError on invalid input
raw.decode('utf-8', errors='replace')     # invalid bytes -> U+FFFD
raw.decode('utf-8', errors='ignore')      # invalid bytes silently dropped
raw.decode('utf-8', errors='surrogateescape')  # round-trippable, for filenames

Prefer failing loudly. `errors='ignore'` converts a data problem into silent
corruption, which is worse. Use `surrogateescape` for filesystem paths, where
you must round-trip whatever the OS gave you.

### `str` methods you will actually use

```text
s.strip() / .lstrip() / .rstrip()      # whitespace or given chars
s.split(",") / .rsplit(",", 1)         # rsplit with maxsplit is underused
s.partition(":")                       # ('before', ':', 'after') -- never raises
"-".join(parts)                        # the ONLY correct way to build from a list
s.startswith(("http://", "https://"))  # accepts a TUPLE of prefixes
s.replace(old, new, count)
s.casefold()                           # aggressive lowercase for comparison
s.removeprefix("v") / .removesuffix(".txt")   # 3.9+, better than slicing
s.encode("utf-8")
```


Two notes. `s.lower()` is for display; `s.casefold()` is for comparison (it
handles the German ß and similar correctly). And `s.strip("abc")` strips *any of
those characters*, not the substring — a very common misreading:

```text
>>> "example.com".strip("moc.")
'example'                # not what most people expect
>>> "example.com".removesuffix(".com")
'example'                # what they meant
```


### f-strings and the format mini-language

In [ ]:
name, value, ratio = "cpu", 1234.5678, 0.8532

f"{name}: {value}"              # cpu: 1234.5678
f"{value:.2f}"                  # 1234.57
f"{value:,.2f}"                 # 1,234.57
f"{value:>12.2f}"               # right-align in 12 cols
f"{value:<12.2f}"               # left-align
f"{value:^12.2f}"               # centre
f"{ratio:.1%}"                  # 85.3%
f"{255:#x}  {255:08b}  {255:o}" # 0xff  11111111  377
f"{value:e}"                    # 1.234568e+03
f"{name!r}"                     # 'cpu'   -- repr, quotes visible
f"{value=}"                     # value=1234.5678   -- debugging gold
f"{value:{width}.{prec}f}"      # nested: width and precision from variables

from datetime import datetime
f"{datetime.now():%Y-%m-%d %H:%M}"

`f"{x=}"` prints both the expression text and its value. It is the single best
replacement for `print("x is", x)` and you should adopt it today.

Use `!r` in every error message and log line. `repr` shows quotes, escapes, and
whitespace — precisely the things that matter when the bug is an empty string or
a trailing space.

**Building strings in a loop:**

In [ ]:
parts = []
for item in items:
    parts.append(transform(item))
result = "".join(parts)                     # correct

result = ""
for item in items:
    result += transform(item)               # O(n^2) in principle

CPython has an optimisation that often makes the second form linear anyway, but
it is fragile (it only applies when the string has a refcount of 1) and it does
not hold on other implementations. `"".join()` is both faster and clearer.

---

## Concept 3. Slicing

Slicing works on every sequence: `str`, `bytes`, `list`, `tuple`, `range`.

In [ ]:
s = "abcdefgh"
s[2]        # 'c'
s[2:5]      # 'cde'      start inclusive, stop EXCLUSIVE
s[:3]       # 'abc'
s[3:]       # 'defgh'
s[-2:]      # 'gh'
s[::2]      # 'aceg'     every 2nd
s[::-1]     # 'hgfedcba' reversed
s[1:6:2]    # 'bdf'

The half-open convention `[start, stop)` gives you three useful invariants:
`len(s[a:b]) == b - a`, `s[:i] + s[i:] == s`, and adjacent slices tile without
overlap or gaps.

**Slices never raise IndexError**, which is a frequent source of silent bugs:

```text
>>> "abc"[10]
IndexError
>>> "abc"[10:20]
''                      # no error, just empty
```


Slice assignment on lists is powerful and worth knowing:

In [ ]:
lst = [1, 2, 3, 4, 5]
lst[1:3] = [9]          # [1, 9, 4, 5]     replace, lengths need not match
lst[::2] = [0, 0, 0]    # extended slice assignment MUST match length
del lst[1:3]
lst[:] = other          # replace CONTENTS in place -- visible to all aliases
lst = other             # rebind -- visible to nobody else  (Module 02!)

That last pair is Module 02 again. `lst[:] = other` mutates; `lst = other`
rebinds.

---

## Concept 5. Hashability

An object is hashable if it has a `__hash__` and its hash never changes. Only
hashable objects can be dict keys or set members.

In [ ]:
hash("abc")            # fine
hash((1, 2))           # fine
hash([1, 2])           # TypeError: unhashable type: 'list'
hash((1, [2]))         # TypeError -- tuple hashes its CONTENTS

**The rule:** immutable built-ins are hashable; mutable ones are not. Your own
classes are hashable by default (by identity), and Module 09 covers what happens
when you define `__eq__`.

Why the restriction? A dict finds a key by its hash. If a key's hash changed
after insertion, the dict would look in the wrong bucket and the entry would be
unreachable — present in memory, invisible to lookup. Forbidding mutable keys
makes that unrepresentable.

Note that hash equality does not imply object equality — collisions exist, and
dicts handle them by comparing with `==` after matching hashes. This is why
`__eq__` and `__hash__` must agree (Module 09).

```text
>>> hash(1) == hash(1.0) == hash(True)
True
>>> {1: "int", 1.0: "float", True: "bool"}
{1: 'bool'}            # all three are equal AND hash equal: one key, last wins
```


That last one is a genuinely surprising result worth staring at.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: Numbers
- Section 2: Strings and bytes: the boundary that matters
- Section 3: Slicing
- Section 4: The four containers, at a glance
- Section 5: Hashability

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

SAMPLE_LINES = [
    "2026-08-01 INFO  user=José action=login",
    "2026-08-01 WARN  user=Müller action=retry café=true",
    "2026-08-01 ERROR user=Ωmega action=crash naïve=yes",
    "2026-08-01 INFO  user=张伟 action=logout",
]

---

## `make_fixtures`

Write the same content in three different encodings.

In [ ]:
def make_fixtures(tmp: Path) -> dict[str, Path]:
    """Write the same content in three different encodings."""
    files = {}
    for name, enc in [("utf8.log", "utf-8"), ("latin1.log", "latin-1"),
                      ("utf16.log", "utf-16")]:
        p = tmp / name
        try:
            p.write_text("\n".join(SAMPLE_LINES), encoding=enc)
        except UnicodeEncodeError:
            # latin-1 cannot represent every character -- that is the point
            p.write_bytes("\n".join(SAMPLE_LINES[:2]).encode("latin-1"))
        files[name] = p
    return files

---

## `count_users_broken`

BUG 1: no encoding specified. BUG 2: reads bytes and str inconsistently.

In [ ]:
def count_users_broken(path: Path) -> dict[str, int]:
    """BUG 1: no encoding specified. BUG 2: reads bytes and str inconsistently.

    TODO: find all the bugs by reading, before running.
    """
    counts: dict[str, int] = {}
    with open(path) as fh:                       # BUG: locale-dependent
        for line in fh:
            for field in line.split():
                if field.startswith("user="):
                    user = field[5:]
                    counts[user] = counts.get(user, 0) + 1
    return counts

---

## `count_users`

Fix count_users_broken. Explicit encoding, explicit error handling.

In [ ]:
def count_users(path: Path, encoding: str = "utf-8") -> dict[str, int]:
    """Fix count_users_broken. Explicit encoding, explicit error handling.

    Decide and justify: should a malformed byte sequence raise, or be replaced?
    Write your answer as a docstring line.
    """
    raise NotImplementedError

---

## `detect_encoding`

Guess a file's encoding well enough to read it.

In [ ]:
def detect_encoding(path: Path) -> str:
    """Guess a file's encoding well enough to read it.

    Strategy, in order:
      1. Check for a BOM (utf-8-sig, utf-16-le, utf-16-be, utf-32). The codecs
         module has the BOM constants.
      2. Try utf-8 strictly. If it decodes, it is almost certainly utf-8 --
         utf-8 is self-validating, and random bytes rarely decode cleanly.
      3. Fall back to a declared default (cp1252 or latin-1). latin-1 NEVER
         fails, because every byte 0-255 maps to some character. Explain in a
         comment why "never fails" is a warning rather than a feature.

    Real code uses charset-normalizer or chardet for this. Implementing the
    simple version once teaches you what those libraries are actually doing.
    """
    raise NotImplementedError

---

## `read_any`

Read a text file of unknown encoding, never raising, never silently

In [ ]:
def read_any(path: Path) -> str:
    """Read a text file of unknown encoding, never raising, never silently
    corrupting. Report on stderr which encoding was chosen."""
    raise NotImplementedError

---

## `safe_truncate`

Truncate text so its UTF-8 encoding is at most max_bytes, WITHOUT

In [ ]:
def safe_truncate(text: str, max_bytes: int, encoding: str = "utf-8") -> str:
    """Truncate text so its UTF-8 encoding is at most max_bytes, WITHOUT
    splitting a character in half.

    This is a real problem: database columns, log fields, and HTTP headers are
    limited in BYTES, while your text is measured in CHARACTERS.

    Naive `text[:max_bytes]` is wrong (wrong unit).
    Naive `text.encode()[:max_bytes].decode()` raises UnicodeDecodeError when it
    cuts mid-character.

    Two correct approaches -- implement either, and name the other in a comment:
      (a) encode, slice, then decode with errors='ignore'
      (b) walk characters accumulating byte lengths until the budget is spent
    Which is faster? Which is clearer? Do they always agree?
    """
    raise NotImplementedError

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    with tempfile.TemporaryDirectory() as td:
        tmp = Path(td)
        files = make_fixtures(tmp)

        counts = count_users(files["utf8.log"])
        assert counts.get("José") == 1, counts
        assert counts.get("张伟") == 1, counts

        assert detect_encoding(files["utf8.log"]).startswith("utf-8")
        assert detect_encoding(files["utf16.log"]).startswith("utf-16")

        text = read_any(files["latin1.log"])
        assert "user=" in text

    assert safe_truncate("café", 4) == "caf", safe_truncate("café", 4)
    assert safe_truncate("café", 5) == "café"  # é is 2 bytes: "café" is 5 bytes
    assert safe_truncate("张伟好", 7) == "张伟"
    assert len(safe_truncate("张伟好", 7).encode("utf-8")) <= 7

    print("all encoding checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.